# N170 Group Analysis

Pre-registered secondary confirmatory analysis (prerec.md §5).
Run after n170_per_subject.ipynb has been executed for every subject.

Statistical plan:
1. **Primary (non-parametric):** Friedman test across the four conditions,
   followed by pairwise Wilcoxon signed-rank post-hoc tests vs control,
   one-tailed (predicted direction: noise makes N170 less negative).
2. **Robustness check (parametric):** Repeated-measures ANOVA with
   Greenhouse-Geisser correction. Agreement between the two strengthens
   the conclusion; disagreement is reported and discussed.

Effect sizes: Cohen's d for paired comparisons with 95 % CIs.

In [ ]:
# ---- CONFIGURE HERE -------------------------------------------------------
DERIVED_DIR = 'data/derived'
ALPHA       = 0.05   # significance level
# ---------------------------------------------------------------------------

In [ ]:
import sys
import os
from pathlib import Path

_here = Path('.').resolve()
repo_root = next(
    (p for p in [_here, _here.parent, _here.parent.parent]
     if (p / 'config.yaml').exists()),
    _here,
)
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print(f'Working directory: {Path.cwd()}')

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
plt.rcParams['figure.dpi'] = 120

from analysis.n170 import (
    load_all_n170_results,
    build_amplitude_dataframe,
    run_group_statistics,
    CONDITIONS,
)
from analysis.plots import (
    plot_n170_group_amplitude,
    plot_n170_group_erp,
    CONDITION_COLORS,
)
print('Imports OK')

## 1. Load results

In [ ]:
results = load_all_n170_results(DERIVED_DIR)
print(f'Found {len(results)} subjects with N170 results:')
for r in results:
    amps = {c: r['per_condition'][c]['n170_amplitude_uv']
            for c in CONDITIONS if c in r['per_condition']}
    amp_str = '  '.join(f'{c}={v:.2f}' for c, v in amps.items())
    print(f'  sub-{r["subject_id"]}  {amp_str}')

if not results:
    raise RuntimeError('No N170 results found. Run n170_per_subject.ipynb first.')

## 2. Descriptive statistics

In [ ]:
df = build_amplitude_dataframe(results)

desc = df.groupby('condition')['n170_amplitude_uv'].agg(['count','mean','std','median'])
desc.columns = ['N', 'Mean (µV)', 'SD (µV)', 'Median (µV)']
desc = desc.round(3).loc[CONDITIONS]   # force display order
print('N170 amplitude by condition (µV):')
print(desc.to_string())

## 3. Subject × condition amplitude table

In [ ]:
table = df.pivot(index='subject', columns='condition', values='n170_amplitude_uv')
table = table[CONDITIONS].round(3)
table.loc['MEAN'] = table.mean().round(3)
table.loc['SD']   = table.iloc[:-1].std().round(3)
table

## 4. Group grand-average ERP (posterior channels)

In [ ]:
fig = plot_n170_group_erp(results)
plt.show()

## 5. Group amplitude by condition

In [ ]:
fig = plot_n170_group_amplitude(results)
plt.show()

## 6. Paired scatter: control vs each noise condition

Each dot is one subject. Points above the diagonal: noise condition is
less negative than control (predicted direction of noise effect).

In [ ]:
noise_conds = [c for c in CONDITIONS if c != 'control']
fig, axes = plt.subplots(1, len(noise_conds),
                         figsize=(5 * len(noise_conds), 4.5),
                         layout='constrained')

for ax, cond in zip(axes, noise_conds):
    ctrl_vals = df[df['condition'] == 'control']['n170_amplitude_uv'].values
    noise_vals = df[df['condition'] == cond]['n170_amplitude_uv'].values
    # Match by subject order (both come from the same sorted results list)
    ctrl_by_subj = (df[df['condition'] == 'control']
                    .set_index('subject')['n170_amplitude_uv'])
    noise_by_subj = (df[df['condition'] == cond]
                     .set_index('subject')['n170_amplitude_uv'])
    common = ctrl_by_subj.index.intersection(noise_by_subj.index)
    c_vals = ctrl_by_subj[common].values
    n_vals = noise_by_subj[common].values

    lims = [min(c_vals.min(), n_vals.min()) - 0.5,
            max(c_vals.max(), n_vals.max()) + 0.5]
    ax.plot(lims, lims, 'k--', linewidth=0.8, alpha=0.5)
    ax.scatter(c_vals, n_vals,
               color=CONDITION_COLORS.get(cond, '#888888'),
               s=40, alpha=0.8, edgecolors='white', linewidths=0.5)
    ax.set_xlabel('Control N170 amplitude (µV)')
    ax.set_ylabel(f'{cond} N170 amplitude (µV)')
    ax.set_title(f'Control vs {cond}  (n={len(common)})')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal')

fig.suptitle('N170 amplitude: control vs noise conditions\n'
             'Points above diagonal = noise more positive (attenuated N170)',
             fontsize=11)
plt.show()

## 7. Pre-registered statistics

Primary: Friedman test + pairwise Wilcoxon vs control (one-tailed).  
Robustness: RM-ANOVA with Greenhouse-Geisser correction.

In [ ]:
stats = run_group_statistics(results)

print(f'N subjects (complete cases): {stats["n_subjects"]}')
print()

# ---- Descriptives --------------------------------------------------------
print('Descriptives (µV):')
print(f'{"Condition":<20} {"Mean":>8} {"SD":>8} {"SEM":>8} {"Median":>8}')
print('-' * 56)
for cond in CONDITIONS:
    d = stats['descriptives'][cond]
    print(f'{cond:<20} {d["mean_uv"]:>8.3f} {d["sd_uv"]:>8.3f} '
          f'{d["sem_uv"]:>8.3f} {d["median_uv"]:>8.3f}')

print()

# ---- Friedman ------------------------------------------------------------
fr = stats['friedman']
sig = 'SIGNIFICANT' if fr['p_value'] < ALPHA else 'not significant'
print(f'Friedman test:  χ²({fr["df"]}) = {fr["statistic"]:.3f},  '
      f'p = {fr["p_value"]:.4f}  [{sig} at α={ALPHA}]')
print()

# ---- Wilcoxon post-hoc ---------------------------------------------------
print('Wilcoxon post-hoc vs control (one-tailed: noise > control, i.e. less negative):')
print(f'{"Condition":<12} {"W":>8} {"p (1-tail)":>12} {"p (Holm)":>12} '
      f'{"Cohen d":>9} {"95% CI":>20} {"Mean Δ":>9}')
print('-' * 90)
for cond, ph in stats['wilcoxon_posthoc_vs_control'].items():
    sig_marker = '*' if ph.get('significant_holm') else ' '
    ci = f'[{ph["ci_low"]:+.3f}, {ph["ci_high"]:+.3f}]'
    print(f'{cond:<12} {ph["wilcoxon_statistic"]:>8.1f} '
          f'{ph["p_one_tailed"]:>12.4f} '
          f'{ph["p_one_tailed_holm"]:>12.4f} '
          f'{ph["cohens_d"]:>9.3f} '
          f'{ci:>20} '
          f'{ph["mean_diff_uv"]:>+9.3f} {sig_marker}')
print(f'  * = Holm-adjusted p < {ALPHA} (one-tailed)')
print()

# ---- RM-ANOVA ------------------------------------------------------------
rm = stats['rm_anova']
if 'error' in rm:
    print(f'RM-ANOVA error: {rm["error"]}')
else:
    print('RM-ANOVA (robustness check):')
    print(f'  F({rm["df_numerator"]:.0f}, {rm["df_denominator"]:.0f}) = {rm["f_statistic"]:.3f},  '
          f'p (uncorrected) = {rm["p_uncorrected"]:.4f}')
    print(f'  Greenhouse-Geisser ε = {rm["greenhouse_geisser_epsilon"]:.3f}')
    print(f'  F({rm["df_numerator_gg"]:.2f}, {rm["df_denominator_gg"]:.2f}) = {rm["f_statistic"]:.3f},  '
          f'p (GG-corrected) = {rm["p_greenhouse_geisser"]:.4f}')
    gg_sig = 'SIGNIFICANT' if rm['p_greenhouse_geisser'] < ALPHA else 'not significant'
    print(f'  GG-corrected result [{gg_sig} at α={ALPHA}]')

## 8. Parametric vs non-parametric agreement

Per prerec §5: agreement between parametric and non-parametric results
strengthens the conclusion; disagreement is reported and discussed.

In [ ]:
fr_sig  = stats['friedman']['p_value'] < ALPHA
rm      = stats['rm_anova']
rm_sig  = (rm.get('p_greenhouse_geisser', 1.0) < ALPHA) if 'error' not in rm else None

print(f'Friedman p = {stats["friedman"]["p_value"]:.4f}  →  {"SIGNIFICANT" if fr_sig else "not sig"}')
if rm_sig is not None:
    print(f'RM-ANOVA (GG) p = {rm["p_greenhouse_geisser"]:.4f}  →  '
          f'{"SIGNIFICANT" if rm_sig else "not sig"}')
    if fr_sig == rm_sig:
        print('\nAgreement between parametric and non-parametric results.')
    else:
        print('\nDISAGREEMENT between parametric and non-parametric. '
              'Investigate distributional assumptions — report both results.')
else:
    print(f'RM-ANOVA error: {rm.get("error")}')

## 9. Effect size forest plot

In [ ]:
noise_conds = list(stats['wilcoxon_posthoc_vs_control'].keys())

fig, ax = plt.subplots(figsize=(7, max(3, len(noise_conds) * 1.2 + 1)),
                       layout='constrained')

y_positions = list(range(len(noise_conds)))
for y, cond in zip(y_positions, noise_conds):
    ph = stats['wilcoxon_posthoc_vs_control'][cond]
    d  = ph['cohens_d']
    lo = ph['ci_low']
    hi = ph['ci_high']
    color = CONDITION_COLORS.get(cond, '#888888')
    ax.errorbar(d, y, xerr=[[d - lo], [hi - d]],
                fmt='o', color=color, markersize=8,
                capsize=5, linewidth=1.5)
    sig = '*' if ph.get('significant_holm') else ''
    ax.text(hi + 0.05, y, f" d={d:.2f}{sig}", va='center', fontsize=9)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(y_positions)
ax.set_yticklabels(noise_conds)
ax.set_xlabel("Cohen's d  (positive = noise condition less negative than control)")
ax.set_title('Effect sizes: noise vs control N170 amplitude\n'
             f'Error bars = 95% CI  |  * = p < {ALPHA} (one-tailed Wilcoxon)')
plt.show()